In [1]:
# Cell 1: Clone repository from git
import subprocess
import os

if not os.path.exists('repo'):
    subprocess.run(['git', 'clone', 'https://github.com/EugenTheMachine/instanseg.git', 'repo'], check=True)


Cloning into 'repo'...


In [2]:
# Cell 2: Setup environment and install required dependencies
import os, sys, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.chdir('/kaggle/working/repo')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'monai', 'fastremap', 'edt', 'albumentations', 'colorcet', 'tifffile', 'opencv-python-headless'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', '.'], check=True)
if '/kaggle/working/repo' not in sys.path:
    sys.path.insert(0, '/kaggle/working/repo')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 808.6/808.6 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 49.8 MB/s eta 0:00:00
Obtaining file:///kaggle/working/repo
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for instanseg-torch (pyproject.toml): started
  Building editable for instanseg-torch (pyproject.toml): f

In [3]:
# Cell 3: Import required modules
import sys
from pathlib import Path
from instanseg import InstanSegModel
import instanseg
print('InstanSeg package loaded from:', instanseg.__file__)


InstanSeg package loaded from: /kaggle/working/repo/instanseg/__init__.py


In [4]:
# Cell 4: Specify path to dataset folder dynamically (resolving to top-level dataset root)
from pathlib import Path
import os

input_dir = Path('/kaggle/input')
data_dir = None

# Search top-level dataset directories under /kaggle/input/
for p in input_dir.glob('*'):
    if p.is_dir():
        subdirs = [x.name.lower() for x in p.iterdir() if x.is_dir()]
        if any(s in subdirs for s in ['train', 'images', 'val', 'labels', 'masks']):
            data_dir = str(p)
            break

# If not found at top level, search recursively
if data_dir is None:
    for p in input_dir.rglob('*'):
        if p.is_dir():
            subdirs = [x.name.lower() for x in p.iterdir() if x.is_dir()]
            if any(s in subdirs for s in ['train', 'images', 'val', 'labels', 'masks']):
                data_dir = str(p)
                break

# Safety check: normalize to root if pointing to a subfolder
if data_dir is not None:
    d_path = Path(data_dir)
    while d_path.name.lower() in ['train', 'val', 'test', 'images', 'labels', 'masks']:
        parent = d_path.parent
        if parent == d_path:
            break
        subdirs = [x.name.lower() for x in parent.iterdir() if x.is_dir()]
        if any(s in subdirs for s in ['train', 'val', 'test', 'images', 'labels', 'masks']):
            d_path = parent
        else:
            break
    data_dir = str(d_path)

if data_dir is None:
    possible_dirs = list(input_dir.glob('*'))
    if possible_dirs:
        data_dir = str(possible_dirs[0])
    else:
        data_dir = '/kaggle/input/overfit-small-dataset'

print('Using DATA_DIR:', data_dir)


Using DATA_DIR: /kaggle/input/livecell-cellseg1-a172


In [5]:
# # Cell 5: Init InstanSegModel and train from scratch for 2 epochs
# model = InstanSegModel()
# metrics = model.train(
#     epochs=20,
#     imgsz=512,
#     batch_size=4,
#     train_data_ratio=1.0,
#     data_dir=data_dir,
#     experiment_name='kaggle_scratch_run'
# )
# print('Scratch training finished with metrics:', metrics)


In [6]:
## Cell 5 V2: train the efficientnet-based model for 20 epochs
# model = InstanSegModel(model_name='instanseg_unet')
# metrics = model.train(
#     epochs=20,
#     imgsz=512,
#     batch_size=4,
#     train_data_ratio=1.0,
#     data_dir=data_dir,
#     experiment_name='kaggle_efficientnetb0_run'
# )
# print('EfficientNet-B0 training finished with metrics:', metrics)


In [7]:
# # Cell 5 V2: train the efficientnet-based model for 20 epochs
# model = InstanSegModel(model_name='efficientunetb0')
# metrics = model.train(
#     epochs=20,
#     imgsz=512,
#     batch_size=4,
#     train_data_ratio=1.0,
#     data_dir=data_dir,
#     experiment_name='kaggle_efficientnetb0_run'
# )
# print('EfficientNet-B0 training finished with metrics:', metrics)


In [8]:
# # Cell 6: Load pre-trained model from checkpoint and resume training for 2 more epochs (total 4 epochs)
# checkpoint_path = '/kaggle/working/repo/runs/kaggle_scratch_run/checkpoints/last.pt'
# resumed_model = InstanSegModel(checkpoint_path)
# resumed_metrics = resumed_model.train(
#     epochs=4,
#     imgsz=128,
#     batch_size=1,
#     train_data_ratio=1.0,
#     data_dir=data_dir,
#     resume=True,
#     experiment_name='kaggle_resume_run'
# )
# print('Resumed training finished with metrics:', resumed_metrics)


In [9]:
# # Cell 7: Init ConvNeXt-based U-Net model and train for 20 epochs in AMP (FP16) mode
# model = InstanSegModel(model_name='convnext_unet')
# metrics = model.train(
#     epochs=20,
#     imgsz=512,
#     batch_size=1,
#     train_data_ratio=1.0,
#     data_dir=data_dir,
#     amp=True,
#     max_seeds=300,
#     experiment_name='kaggle_convnext_unet_run'
# )
# print('ConvNeXt U-Net training finished with metrics:', metrics)

In [10]:
# Cell 8: Init Improved U-Net model and train for 20 epochs
# model = InstanSegModel(model_name='improved_unet')
# metrics = model.train(
#     epochs=20,
#     imgsz=512,
#     batch_size=4,
#     train_data_ratio=1.0,
#     data_dir=data_dir,
#     amp=False,
#     experiment_name='kaggle_improved_unet_run'
# )
# print('Improved U-Net training finished with metrics:', metrics)

In [11]:
# # Cell 9: Init MedNeXt-2D model (shallow option) and train for 20 epochs
# model = InstanSegModel(model_name='instanseg_unet_v2')
# metrics = model.train(
#     epochs=20,
#     imgsz=512,
#     batch_size=4,
#     train_data_ratio=1.0,
#     data_dir=data_dir,
#     amp=True,
#     experiment_name='kaggle_instanseg_unet_v2_run'
# )
# print('InstanSeg U-Net training finished with metrics:', metrics)

In [12]:
# Cell 10: Init MobileNetV2 U-Net model and train for 20 epochs
# This is a lightweight efficient architecture with ~60% fewer parameters than InstanSeg_UNet
# Uses pretrained MobileNetV2 encoder from ImageNet for better feature extraction
model = InstanSegModel(model_name='instanseg_unet')
metrics = model.train(
    epochs=200,
    imgsz=512,
    batch_size=4,
    train_data_ratio=1.0,
    data_dir=data_dir,
    amp=True,
    max_seeds=300,
    experiment_name='kaggle_instanseg_unet_run',
    patience=10
)
print('MobileNetV2 U-Net training finished with metrics:', metrics)

[2026-11-19 17:23:30] Data loaded successfully
[2026-11-19 17:23:30] Augmentation pipeline initialized
[2026-11-19 17:23:30] Config: {'meta': {'data_dir': '/kaggle/input/livecell-cellseg1-a172', 'experiment_name': 'kaggle_instanseg_unet_run'}, 'preprocessing': {'imgsz': 512, 'train_data_ratio': 1.0, 'seed': 42, 'image_modality': 'auto'}, 'training': {'epochs': 200, 'patience': 10, 'batch_size': 4, 'val_ratio': 0.2, 'test_ratio': 0.2, 'val_interval': 1, 'warmup_epochs': 2, 'resume': False, 'amp': True, 'max_seeds': 300}, 'model': {'model_name': 'instanseg_unet', 'num_classes': 2, 'freeze_backbone_epochs': 0, 'pretrained_backbone': True, 'dropout': 0.0, 'embedding_mode': 'center-seed'}, 'optimizer': {'type': 'AdamW', 'learning_rate': 0.001, 'weight_decay': 0.0001, 'momentum': 0.9, 'backbone_lr_mult': 0.1}}
Generating instanseg_unet
[2026-11-19 17:23:30] AMP (FP16) mixed precision training mode enabled.
[2026-11-19 17:23:30] Model initialized from scratch
[2026-11-19 17:23:30] Warmup Epoc

  0%|          | 0/77 [00:00<?, ?it/s]

Validation:   0%|          | 0/16 [00:00<?, ?it/s]

/kaggle/working/repo/instanseg/utils/pytorch_utils.py:303: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  intersection = torch.sparse.mm(onehot1, onehot2.T).to_dense()


[2026-11-19 17:24:34] Epoch 1 Results - Train Loss: 7.2467 | Val Loss: 6.705251634120941 | Precision: 0.0144 | Recall: 0.0003 | Accuracy: 0.0003 | F1: 0.0006
[2026-11-19 17:24:34] Warmup Epoch 2/2: head=0.001000
[2026-11-19 17:24:34] EPOCH 2/200


  0%|          | 0/77 [00:00<?, ?it/s]

... (195 outputs left out)

Validation:   0%|          | 0/16 [00:00<?, ?it/s]

[2026-11-19 18:32:56] Epoch 67 Results - Train Loss: 2.3871 | Val Loss: 2.471043512225151 | Precision: 0.6663 | Recall: 0.6589 | Accuracy: 0.4954 | F1: 0.6626
[2026-11-19 18:32:57] Early stopping triggered at epoch 67


Test Evaluation:   0%|          | 0/16 [00:00<?, ?it/s]

[2026-11-19 18:33:09] TEST RESULTS - Loss: 2.3177 | Precision: 0.7427 | Recall: 0.7072 | Accuracy: 0.5681 | F1: 0.7246
MobileNetV2 U-Net training finished with metrics: [{'epoch': 1, 'train_loss': 7.246685981750488, 'val_loss': 6.705251634120941, 'precision': 0.014354066985639066, 'recall': 0.0002813731007315674, 'accuracy': 0.0002760397497239577, 'f1': 0.0005519271456167735}, {'epoch': 2, 'train_loss': 6.65671443939209, 'val_loss': 6.074823766946793, 'precision': 0.0003220611916263986, 'recall': 9.404683532399047e-05, 'accuracy': 7.279609812913974e-05, 'f1': 0.00014558159848595032}, {'epoch': 3, 'train_loss': 6.213691234588623, 'val_loss': 6.282737731933594, 'precision': 0.3432835820895494, 'recall': 0.38497303328993504, 'accuracy': 0.22169861839991314, 'f1': 0.36293503988778664}, {'epoch': 4, 'train_loss': 6.058742046356201, 'val_loss': 5.895946055650711, 'precision': 0.4547690842723341, 'recall': 0.5170516059184364, 'accuracy': 0.3191868560289595, 'f1': 0.4839145486785423}, {'epoch'